# Transaction Data Analysis Pipeline

End-to-end data analysis: EDA → Cleaning → Validation → Feature Engineering → Visualization.

---
## 1. Exploratory Data Analysis (EDA)

Goal: Understand the dataset structure, data types, descriptive statistics, and missing value patterns before any cleaning.

In [ ]:
import pandas as pd

# ── Load Data ──
df = pd.read_csv("module3_4_full_demo_dataset.csv")

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"\nColumn Names:\n{list(df.columns)}")

print("\n" + "=" * 50)
print("DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n" + "=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
print(df.describe(include='all').T)

print("\n" + "=" * 50)
print("MISSING VALUES")
print("=" * 50)
missing = pd.DataFrame({
    'Count'  : df.isnull().sum(),
    'Percent': (df.isnull().mean() * 100).round(2)
})
print(missing[missing['Count'] > 0].to_string() or "No missing values found.")

print("\n" + "=" * 50)
print("CATEGORICAL DISTRIBUTIONS")
print("=" * 50)
for col in ['region', 'product_category', 'high_value']:
    if col in df.columns:
        print(f"\n{col}:\n{df[col].value_counts().to_string()}")


---
## 2. Data Cleaning

Goal: Fix data types, handle missing values, standardize categories, remove invalid rows, recalculate derived columns, and eliminate duplicates.

In [ ]:
import pandas as pd
import numpy as np

# ── Load Raw Data ──
df = pd.read_csv("module3_4_full_demo_dataset.csv")

# ── Fix Data Types ──
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
df['units_sold']       = pd.to_numeric(df['units_sold'],  errors='coerce')
df['unit_price']       = pd.to_numeric(df['unit_price'],  errors='coerce')

# ── Drop rows with unparseable dates (cannot be used in time-series) ──
before = len(df)
df = df.dropna(subset=['transaction_date'])
print(f"Dropped {before - len(df)} rows with invalid dates.")

# ── Drop rows with negative units or price (logically invalid) ──
before = len(df)
df = df[(df['units_sold'] >= 0) & (df['unit_price'] >= 0)]
print(f"Dropped {before - len(df)} rows with negative values.")

# ── Impute remaining missing values ──
# Region: use mode (most frequent region)
df['region']           = df['region'].fillna(df['region'].mode()[0])
# Category: label as 'unknown' when category is unidentifiable
df['product_category'] = df['product_category'].fillna("unknown")
# Numeric: use median (robust to outliers)
df['units_sold']  = df['units_sold'].fillna(df['units_sold'].median())
df['unit_price']  = df['unit_price'].fillna(df['unit_price'].median())

# ── Standardize text ──
df['region'] = df['region'].str.strip().str.title()

def clean_category(x):
    """Map noisy category strings to canonical labels."""
    if pd.isna(x):
        return "unknown"
    x = x.lower().strip()
    if "cloth" in x:    return "clothes"
    if "elect" in x:    return "electronics"
    if "beaut" in x:    return "beauty"
    return x

df['product_category'] = df['product_category'].apply(clean_category)

# ── Recalculate derived columns so they stay consistent ──
df['revenue']    = df['units_sold'] * df['unit_price']
df['high_value'] = (df['revenue'] >= 1000).astype(int)

# ── Remove duplicates on transaction_id ──
id_col = 'transaction_id' if 'transaction_id' in df.columns else None
before = len(df)
df = df.drop_duplicates(subset=[id_col]) if id_col else df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows.")

# ── Save ──
df.to_csv("cleaned_transaction_dataset.csv", index=False)
print(f"\n✅ Cleaning complete — final shape: {df.shape}")


---
## 3. Data Quality Validation

Goal: Verify the cleaned dataset is logically sound. Each check targets a specific business rule and **only prints the failing records** so issues are immediately actionable.

In [ ]:
import pandas as pd
import numpy as np

THRESHOLD = 1000  # high-value revenue threshold

# ── Load both versions ──
raw     = pd.read_csv("module3_4_full_demo_dataset.csv")
cleaned = pd.read_csv("cleaned_transaction_dataset.csv",
                      parse_dates=['transaction_date'])

# Fix raw types for fair comparison
raw['transaction_date'] = pd.to_datetime(raw['transaction_date'], errors='coerce')
raw['units_sold']       = pd.to_numeric(raw['units_sold'],  errors='coerce')
raw['unit_price']       = pd.to_numeric(raw['unit_price'],  errors='coerce')
raw['revenue']          = pd.to_numeric(raw.get('revenue',  pd.Series(dtype=float)), errors='coerce')

def check(title, before_mask, after_mask, before_df, after_df, cols):
    """Print a before/after comparison for one validation rule."""
    b_fail = before_df[before_mask]
    a_fail = after_df[after_mask]

    b_n, a_n = len(b_fail), len(a_fail)
    fixed    = b_n - a_n
    b_status = f"❌  {b_n} issue(s)" if b_n else "✅  0 issues"
    a_status = f"✅  0 issues" if a_n == 0 else f"❌  {a_n} issue(s)"

    print(f"\n{'═'*58}")
    print(f"  {title}")
    print(f"{'─'*58}")
    print(f"  BEFORE cleaning : {b_status}")
    if b_n:
        print(b_fail[cols].head(5).to_string(index=False))
        if b_n > 5:
            print(f"  ... ({b_n - 5} more rows)")
    print(f"  AFTER  cleaning : {a_status}")
    if a_n:
        print(a_fail[cols].head(5).to_string(index=False))
    if b_n and a_n == 0:
        print(f"  → Fixed {fixed} row(s) ✔")


# ══════════════════════════════════════════════
# CHECK 1 — Missing Values
# ══════════════════════════════════════════════
print("\n" + "═"*58)
print("  CHECK 1 — Missing Values per Column")
print("─"*58)

raw_miss     = raw.isnull().sum()
cleaned_miss = cleaned.isnull().sum()

miss_report = pd.DataFrame({
    'Before': raw_miss,
    'After' : cleaned_miss,
    'Fixed' : raw_miss - cleaned_miss
})
miss_report = miss_report[miss_report[['Before','After']].max(axis=1) > 0]

if miss_report.empty:
    print("  No missing values in either version.")
else:
    print(miss_report.to_string())

total_before = raw_miss.sum()
total_after  = cleaned_miss.sum()
print(f"\n  Total missing cells  →  Before: {total_before}  |  After: {total_after}  |  Fixed: {total_before - total_after}")


# ══════════════════════════════════════════════
# CHECK 2 — Negative Values
# ══════════════════════════════════════════════
neg_cols = ['units_sold','unit_price']
raw_neg_mask     = (raw['units_sold'] < 0) | (raw['unit_price'] < 0)
cleaned_neg_mask = (cleaned['units_sold'] < 0) | (cleaned['unit_price'] < 0) | (cleaned['revenue'] < 0)

check(
    "CHECK 2 — Negative units / price / revenue",
    raw_neg_mask, cleaned_neg_mask,
    raw, cleaned,
    ['units_sold','unit_price']
)


# ══════════════════════════════════════════════
# CHECK 3 — Revenue Formula Consistency
# ══════════════════════════════════════════════
if 'revenue' in raw.columns:
    raw_calc      = raw['units_sold'] * raw['unit_price']
    raw_rev_mask  = ~raw['revenue'].round(4).eq(raw_calc.round(4))
else:
    raw_rev_mask  = pd.Series([False] * len(raw))

cl_calc       = cleaned['units_sold'] * cleaned['unit_price']
cl_rev_mask   = ~cleaned['revenue'].round(4).eq(cl_calc.round(4))

check(
    "CHECK 3 — Revenue = units_sold × unit_price",
    raw_rev_mask, cl_rev_mask,
    raw, cleaned,
    ['units_sold','unit_price','revenue']
)


# ══════════════════════════════════════════════
# CHECK 4 — high_value Flag Accuracy
# ══════════════════════════════════════════════
if 'high_value' in raw.columns and 'revenue' in raw.columns:
    raw_hv_mask = (
        ((raw['revenue'] >= THRESHOLD) & (raw['high_value'] == 0)) |
        ((raw['revenue'] <  THRESHOLD) & (raw['high_value'] == 1))
    )
else:
    raw_hv_mask = pd.Series([False] * len(raw))

cl_hv_mask = (
    ((cleaned['revenue'] >= THRESHOLD) & (cleaned['high_value'] == 0)) |
    ((cleaned['revenue'] <  THRESHOLD) & (cleaned['high_value'] == 1))
)

check(
    "CHECK 4 — high_value flag vs revenue threshold",
    raw_hv_mask, cl_hv_mask,
    raw, cleaned,
    ['revenue','high_value']
)


# ══════════════════════════════════════════════
# CHECK 5 — Duplicate Rows
# ══════════════════════════════════════════════
id_col = 'transaction_id' if 'transaction_id' in raw.columns else None

if id_col:
    raw_dup_mask     = raw.duplicated(subset=[id_col], keep=False)
    cleaned_dup_mask = cleaned.duplicated(subset=[id_col], keep=False)
    check(
        "CHECK 5 — Duplicate transaction IDs",
        raw_dup_mask, cleaned_dup_mask,
        raw, cleaned,
        [id_col]
    )
else:
    raw_dup_mask     = raw.duplicated(keep=False)
    cleaned_dup_mask = cleaned.duplicated(keep=False)
    check(
        "CHECK 5 — Duplicate rows",
        raw_dup_mask, cleaned_dup_mask,
        raw, cleaned,
        list(raw.columns[:3])
    )


# ══════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════
print(f"\n{'═'*58}")
print("  VALIDATION SUMMARY")
print(f"{'─'*58}")
print(f"  {'Check':<35} {'Before':>8}  {'After':>8}")
print(f"  {'─'*35} {'─'*8}  {'─'*8}")

checks = [
    ("Missing values (total cells)",  total_before,             total_after),
    ("Negative value rows",           raw_neg_mask.sum(),       cleaned_neg_mask.sum()),
    ("Revenue mismatch rows",         raw_rev_mask.sum(),       cl_rev_mask.sum()),
    ("high_value flag errors",        raw_hv_mask.sum(),        cl_hv_mask.sum()),
    ("Duplicate rows",                raw_dup_mask.sum(),       cleaned_dup_mask.sum()),
]

all_clean = True
for name, before, after in checks:
    icon = "✅" if after == 0 else "⚠️ "
    if after > 0:
        all_clean = False
    print(f"  {icon} {name:<33} {before:>8,}  {after:>8,}")

print(f"{'─'*58}")
print(f"  Dataset rows     →  Before: {len(raw):,}  |  After: {len(cleaned):,}")
print(f"{'═'*58}")
print("  ✅  ALL CHECKS PASSED — data is ready." if all_clean else "  ⚠️   Some issues remain — review above.")
print("═"*58)


---
## 4. Feature Engineering

Goal: Derive business-relevant features that reveal purchasing behaviour and seasonal trends.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_transaction_dataset.csv",
                 parse_dates=['transaction_date'])

# ── Time-based features ──
df['month']       = df['transaction_date'].dt.month
df['month_name']  = pd.Categorical(
    df['transaction_date'].dt.month_name(),
    categories=['January','February','March','April','May','June',
                'July','August','September','October','November','December'],
    ordered=True
)
df['quarter']     = df['transaction_date'].dt.to_period('Q').astype(str)
df['day_of_week'] = df['transaction_date'].dt.day_name()
# Weekend = Friday or Saturday (adjust if needed for your locale)
df['is_weekend']  = df['day_of_week'].isin(['Friday','Saturday']).astype(int)

# ── Revenue / pricing features ──
# price_per_unit: cross-check with unit_price; surface any anomalies
df['price_per_unit'] = (df['revenue'] / df['units_sold'].replace(0, np.nan)).round(4)

# Discount detection: if actual revenue < expected → discount applied
df['expected_revenue'] = (df['units_sold'] * df['unit_price']).round(4)
df['discount']         = (df['expected_revenue'] - df['revenue']).clip(lower=0)
df['discount_pct']     = np.where(
    df['expected_revenue'] > 0,
    (df['discount'] / df['expected_revenue']).round(4),
    0
)

# ── Customer-level aggregates ──
df['customer_transactions']  = df.groupby('customer_id')['customer_id'].transform('count')
df['customer_total_spent']   = df.groupby('customer_id')['revenue'].transform('sum').round(2)
df['customer_avg_spent']     = (df['customer_total_spent'] / df['customer_transactions']).round(2)

# ── Dynamic high-value flag (top-25% by revenue) ──
p75 = df['revenue'].quantile(0.75)
df['high_value_dynamic'] = (df['revenue'] >= p75).astype(int)

print(f"Features added: {df.shape[1]} total columns")
print("\nNew columns preview:")
new_cols = ['month','month_name','quarter','day_of_week','is_weekend',
            'price_per_unit','discount','discount_pct',
            'customer_transactions','customer_total_spent','customer_avg_spent',
            'high_value_dynamic']
print(df[new_cols].head(5).to_string(index=False))

df.to_csv("featured_transaction_dataset.csv", index=False)
print("\n✅ Feature engineering complete.")


---
## 5. Business Insights & Visualizations

Goal: Communicate five key findings to stakeholders using clear, professional charts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Styling ──
plt.rcParams.update({
    'figure.dpi'       : 110,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'font.family'      : 'sans-serif',
})
PALETTE = ['#2D6A9F', '#E07B39', '#3A9E6C', '#9B59B6', '#E74C3C']

df = pd.read_csv("featured_transaction_dataset.csv",
                 parse_dates=['transaction_date'])

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle("Transaction Data — Business Insights Dashboard", fontsize=16, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

# ── 1. Revenue by Region ──
ax = axes_flat[0]
region_rev = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
bars = ax.bar(region_rev.index, region_rev.values, color=PALETTE[:len(region_rev)])
ax.bar_label(bars, fmt='${:,.0f}', fontsize=8, padding=3)
ax.set_title("Revenue by Region")
ax.set_xlabel("Region")
ax.set_ylabel("Total Revenue ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=30)

top_region = region_rev.idxmax()
print(f"Insight 1: '{top_region}' generates the highest total revenue.")

# ── 2. Units Sold by Product Category ──
ax = axes_flat[1]
cat_sales = df.groupby('product_category')['units_sold'].sum().sort_values(ascending=False)
bars = ax.bar(cat_sales.index, cat_sales.values, color=PALETTE[:len(cat_sales)])
ax.bar_label(bars, fmt='{:,.0f}', fontsize=8, padding=3)
ax.set_title("Units Sold by Product Category")
ax.set_xlabel("Category")
ax.set_ylabel("Units Sold")
ax.tick_params(axis='x', rotation=30)

top_cat = cat_sales.idxmax()
print(f"Insight 2: '{top_cat}' leads in units sold.")

# ── 3. Monthly Revenue Trend ──
ax = axes_flat[2]
monthly = df.groupby('month')['revenue'].sum()
ax.plot(monthly.index, monthly.values, marker='o', color=PALETTE[0], linewidth=2)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color=PALETTE[0])
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticks(monthly.index)

peak_month = monthly.idxmax()
print(f"Insight 3: Month {peak_month} had the highest revenue.")

# ── 4. Unit Price vs Units Sold ──
ax = axes_flat[3]
sample = df[['unit_price','units_sold']].dropna().sample(min(500, len(df)), random_state=42)
ax.scatter(sample['unit_price'], sample['units_sold'],
           alpha=0.4, s=20, color=PALETTE[2], edgecolors='none')
corr = df[['unit_price','units_sold']].corr().iloc[0, 1]
ax.set_title(f"Unit Price vs Units Sold  (r = {corr:.2f})")
ax.set_xlabel("Unit Price ($)")
ax.set_ylabel("Units Sold")

direction = "negative" if corr < 0 else "positive"
print(f"Insight 4: Correlation is {corr:.2f} ({direction} relationship).")

# ── 5. High-Value vs Low-Value Customers ──
ax = axes_flat[4]
hv = df.groupby('customer_id')['high_value_dynamic'].max().value_counts()
labels = ['Low-Value', 'High-Value']
colors = [PALETTE[4], PALETTE[0]]
wedges, texts, autotexts = ax.pie(
    hv.values, labels=labels, autopct='%1.1f%%',
    colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(10)
ax.set_title("Customer Value Distribution")

hv_pct = (hv.get(1, 0) / hv.sum() * 100)
print(f"Insight 5: {hv_pct:.1f}% of customers qualify as high-value (top-25% revenue).")

# ── Hide unused 6th panel ──
axes_flat[5].set_visible(False)

plt.tight_layout()
plt.savefig("dashboard.png", bbox_inches='tight', dpi=130)
plt.show()
print("\n✅ Dashboard saved as dashboard.png")
